# ChatGPT — Forced vs. Auto Web-Search: Pilot Sensitivity Analysis

**Decision memo / evidence — should the full scale-up *force* web-search on every ChatGPT call?**

This compares the **same 10 queries × 3 runs = 30 ChatGPT cells** under two regimes:

- **Auto** — ChatGPT decides whether to call `web_search` (the main pilot's default behaviour).
- **Forced** — `web_search` is forced on every call (`openai_forced_search_adapter`).

Both regimes are scored with the *same* NLI attribution pipeline
(`src.nli_attribution.attribute_cell_nli`, DeBERTa), so PAWC / AIS are directly comparable.

| layer | auto | forced |
|---|---|---|
| gold | `data/pilot/gold/nli_pilot.parquet` (ChatGPT slice) | `data/gold/forced_pilot_chatgpt.parquet` |

> **Scale:** n = 10 queries (7 vs 27 cited cells). Directional, *not* confirmatory — the forced slice is a sensitivity smoke test, not a powered comparison.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# Resolve repo root whether this runs from notebooks/ or the repo root
here = Path.cwd()
ROOT = next((p for p in [here, *here.parents]
             if (p / 'data' / 'gold' / 'forced_pilot_chatgpt.parquet').exists()), here)

forced = pd.read_parquet(ROOT / 'data' / 'gold' / 'forced_pilot_chatgpt.parquet').copy()
auto   = pd.read_parquet(ROOT / 'data' / 'pilot' / 'gold' / 'nli_pilot.parquet')

# Restrict the auto pilot to ChatGPT on the SAME 10 query_ids the forced slice covers
fq   = sorted(forced['query_id'].unique())
auto = auto[(auto['engine'] == 'chatgpt') & (auto['query_id'].isin(fq))].copy()

# ais_rate is None when a cell has no fetched source -> numeric coercion for means
for df in (forced, auto):
    df['ais_rate_num'] = pd.to_numeric(df['ais_rate'], errors='coerce')

assert len(auto) == 30 and len(forced) == 30, (len(auto), len(forced))
assert set(auto['query_id']) == set(forced['query_id'])
print(f'aligned cells: auto={len(auto)}  forced={len(forced)}  |  {len(fq)} query_ids x 3 runs')

aligned cells: auto=30  forced=30  |  10 query_ids x 3 runs


## 1. Coverage — forcing search sharply increases *how often* ChatGPT cites

The headline effect. Forcing the tool flips ChatGPT from *mostly not searching* to citing on
almost every call — while answer **length is essentially unchanged** (~4 sentences, ~56 words).

In [2]:
metrics = ['n_sentences', 'n_response_words', 'n_sources_cited', 'n_sources_fetched_ok',
           'ais_supported_sentences', 'ais_rate_num', 'pawc_total']
coverage = pd.DataFrame({
    'auto_mean':   [auto[m].mean()   for m in metrics],
    'forced_mean': [forced[m].mean() for m in metrics],
}, index=metrics)
coverage['delta'] = coverage['forced_mean'] - coverage['auto_mean']

n_auto_cited, n_forced_cited = (auto['n_sources_cited'] >= 1).sum(), (forced['n_sources_cited'] >= 1).sum()
print(f'cells with >=1 citation:  auto={n_auto_cited}/30 ({n_auto_cited/30:.0%})   '
      f'forced={n_forced_cited}/30 ({n_forced_cited/30:.0%})')
coverage.round(3)

cells with >=1 citation:  auto=7/30 (23%)   forced=27/30 (90%)


,auto_mean,forced_mean,delta
n_sentences,3.933,4.133,0.200
n_response_words,55.133,56.800,1.667
n_sources_cited,0.267,1.033,0.767
n_sources_fetched_ok,0.267,0.533,0.267
ais_supported_sentences,0.733,1.033,0.300
ais_rate_num,0.111,0.160,0.049
pawc_total,6.910,11.751,4.841


## 2. Quality, *conditional on actually citing* — forcing makes attribution worse

The trade-off. Restricting to cells that *do* cite, the auto regime has far higher per-sentence
support (AIS) and PAWC, and **every** auto source fetches — whereas **~half of forced citations
point to URLs that don't even fetch**, which mechanically depresses NLI support.

In [3]:
ac = auto[auto['n_sources_cited'] >= 1]
fc = forced[forced['n_sources_cited'] >= 1]
quality = pd.DataFrame({
    'auto_cited':   [ac['ais_rate_num'].mean(), ac['pawc_total'].mean(),
                     ac['n_sources_fetched_ok'].sum() / ac['n_sources_cited'].sum(),
                     ac['ais_supported_sentences'].mean()],
    'forced_cited': [fc['ais_rate_num'].mean(), fc['pawc_total'].mean(),
                     fc['n_sources_fetched_ok'].sum() / fc['n_sources_cited'].sum(),
                     fc['ais_supported_sentences'].mean()],
}, index=['ais_rate', 'pawc_total', 'source_fetch_ok_rate', 'ais_supported_sentences'])
print(f'cited cells:  auto n={len(ac)}   forced n={len(fc)}')
quality.round(3)

cited cells:  auto n=7   forced n=27


,auto_cited,forced_cited
ais_rate,0.476,0.178
pawc_total,29.614,13.057
source_fetch_ok_rate,1.000,0.516
ais_supported_sentences,3.143,1.148


## 3. Paired view — forced − auto on each identical cell

Citation *counts* rise in 21/30 cells, but PAWC/AIS are unchanged in the majority (ties): the
aggregate AIS gain is driven by going from *no citation at all* to *a weak citation*, not by
better-supported answers.

In [4]:
m = auto.merge(forced, on=['query_id', 'run_index'], suffixes=('_auto', '_forced'))
rows = []
for col in ['n_sources_cited', 'pawc_total', 'ais_rate_num', 'n_sentences']:
    d = m[f'{col}_forced'] - m[f'{col}_auto']
    rows.append([col, d.mean(), int((d > 0).sum()), int((d < 0).sum()), int((d == 0).sum())])
paired = pd.DataFrame(rows, columns=['metric', 'mean_delta', 'forced>auto', 'forced<auto', 'tie']).set_index('metric')
paired.round(3)

,mean_delta,forced>auto,forced<auto,tie
metric,,,,
n_sources_cited,0.767,21,1,8
pawc_total,4.841,8,4,18
ais_rate_num,0.049,7,4,19
n_sentences,0.200,15,8,7


## 4. Sanity check vs. the pre-computed search/citation CSV

Confirms the gold-derived numbers match `data/pilot/openai_forced_vs_auto.csv` (independent
record of search flags + citation counts).

In [5]:
csv = pd.read_csv(ROOT / 'data' / 'pilot' / 'openai_forced_vs_auto.csv')
print('sanity vs data/pilot/openai_forced_vs_auto.csv')
print(f"  auto  searched rate   : csv {csv['auto_searched'].mean():.0%}   vs  gold cited {(auto['n_sources_cited']>=1).mean():.0%}")
print(f"  forced searched rate  : csv {csv['forced_searched'].mean():.0%}  (gold forced cited {(forced['n_sources_cited']>=1).mean():.0%})")
print(f"  mean auto   citations : csv {csv['auto_n_citations'].mean():.3f}  vs  gold {auto['n_sources_cited'].mean():.3f}")
print(f"  mean forced citations : csv {csv['forced_n_citations'].mean():.3f}  vs  gold {forced['n_sources_cited'].mean():.3f}")

sanity vs data/pilot/openai_forced_vs_auto.csv
  auto  searched rate   : csv 23%   vs  gold cited 23%
  forced searched rate  : csv 100%  (gold forced cited 90%)
  mean auto   citations : csv 0.267  vs  gold 0.267
  mean forced citations : csv 1.033  vs  gold 1.033


## Bottom line — evidence against forcing web-search in the scale-up

- **Coverage rises a lot:** answers with ≥1 citation go **23% → 90%**; mean citations **0.27 → 1.03**.
- **Per-citation quality drops:** among cited cells, AIS **0.48 → 0.18**, PAWC **29.6 → 13.1**, and
  source fetch-OK rate **1.00 → 0.52**. Forcing buys *quantity*, not *quality*.
- **No fluency cost either way:** answer length is flat (~4 sentences / ~56 words), so the change is
  purely in citation behaviour.

**Implication for the full run:** forcing `web_search` would inflate citation *coverage* without
improving — and on a per-citation basis degrading — attribution. The aggregate AIS/PAWC gain is a
coverage artifact (citing where it previously cited nothing), roughly half of it unfetchable URLs
that can't be NLI-verified. On this evidence, **a forced-search scale-up is not justified**; if run
at all it should fix the forced fetch-OK rate first and be reported as coverage, not quality.

*Caveats:* n = 10 queries (7 vs 27 cited cells), directional only; the forced AIS drop is partly an
artifact of unfetchable sources (a sentence can't be entailed against a page we couldn't retrieve).